# YogaLens API — Google Colab Backend
### Run all cells top to bottom. Cell 9 gives you the public URL to paste into yogalens.html

**What this does:**
- Loads your trained model from Google Drive
- Creates a FastAPI server
- Opens a public ngrok tunnel
- You paste the URL into your Web UI

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## Cell 1 — Install dependencies

In [2]:
!pip install fastapi uvicorn python-multipart pyngrok nest-asyncio mediapipe==0.10.13 albumentations -q
print('All dependencies installed!')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.6/35.6 MB 36.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 294.9/294.9 kB 16.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
grpcio-status 1.71.2 requires protobuf<6.0dev,>=5.26.1, but you have protobuf 4.25.8 which is incompatible.
ydf 0.15.0 requires protobuf<7.0.0,>=5.29.1, but you have protobuf 4.25.8 which is incompatible.
grain 0.2.16 requires protobuf>=5.28.3, but you have protobuf 4.25.8 which is incompatible.
opentelemetry-proto 1.38.0 requires protobuf<7.0,>=5.0, but you have protobuf 4.25.8 which is incompatible.
All dependencies installed!


## Cell 2 — Mount Google Drive

In [3]:
from google.colab import drive
drive.mount('/content/drive')
print('Drive mounted!')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive mounted!


## Cell 3 — CONFIGURATION (EDIT THIS CELL ONLY)

In [4]:
from pathlib import Path

# ── EDIT THESE TWO LINES ONLY ─────────────────────────────────────────────

# 1. Path to your .pth model on Google Drive
MODEL_PATH = Path('/content/drive/MyDrive/Yoga-pose-detection ML project/yoga_model_1.pth')

# 2. Free ngrok token from https://dashboard.ngrok.com
NGROK_TOKEN = '3BJdSzfRPl3LVzVBRlCekMgcuX0_5cEbuC2dpYyoFFLdBgFcF'

# ── DO NOT EDIT BELOW ─────────────────────────────────────────────────────
CLASS_NAMES = [
    'adho mukha svanasana', 'virabhadrasana i', 'virabhadrasana ii',
    'vriksasana', 'bhujangasana', 'balasana', 'dandasana', 'chaturanga dandasana'
]
IMG_SIZE = 128
KP_DIM   = 99

print(f'Model path   : {MODEL_PATH}')
print(f'Model exists : {MODEL_PATH.exists()}')
print(f'Classes      : {CLASS_NAMES}')
if not MODEL_PATH.exists():
    print('WARNING: Model file not found — check your MODEL_PATH!')

Model path   : /content/drive/MyDrive/Yoga-pose-detection ML project/yoga_model_1.pth
Model exists : True
Classes      : ['adho mukha svanasana', 'virabhadrasana i', 'virabhadrasana ii', 'vriksasana', 'bhujangasana', 'balasana', 'dandasana', 'chaturanga dandasana']


## Cell 4 — Imports & GPU check

In [5]:
import os, io, cv2, urllib.request
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
import albumentations as A
from albumentations.pytorch import ToTensorV2
import mediapipe as mp

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device : {DEVICE}')
if DEVICE.type == 'cuda':
    print(f'GPU    : {torch.cuda.get_device_name(0)}')

Device : cpu


## Cell 5 — Model architecture (matches your training code)

In [6]:
import torch.nn as nn
import torch

class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch), nn.GELU(),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch), nn.GELU(),
        )
        self.shortcut = nn.Conv2d(in_ch, out_ch, 1, bias=False) if in_ch != out_ch else nn.Identity()
        self.pool = nn.MaxPool2d(2)
    def forward(self, x):
        return self.pool(self.block(x) + self.shortcut(x))

class SpatialAttention(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv = nn.Conv2d(2, 1, kernel_size=7, padding=3, bias=False)
    def forward(self, x):
        avg = x.mean(dim=1, keepdim=True)
        mx  = x.max(dim=1, keepdim=True).values
        return x * torch.sigmoid(self.conv(torch.cat([avg, mx], dim=1)))

class ChannelAttention(nn.Module):
    def __init__(self, channels, reduction=8):
        super().__init__()
        self.fc = nn.Sequential(
            nn.AdaptiveAvgPool2d(1), nn.Flatten(),
            nn.Linear(channels, channels // reduction), nn.ReLU(),
            nn.Linear(channels // reduction, channels), nn.Sigmoid()
        )
    def forward(self, x):
        return x * self.fc(x).view(x.size(0), x.size(1), 1, 1)

class CustomCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.stem        = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1, bias=False),
            nn.BatchNorm2d(32), nn.GELU()
        )
        self.block1      = ConvBlock(32,  64)
        self.block2      = ConvBlock(64,  128)
        self.block3      = ConvBlock(128, 256)
        self.block4      = ConvBlock(256, 256)
        self.spatial_att = SpatialAttention()    # ← MUST be spatial_att
        self.channel_att = ChannelAttention(256) # ← MUST be channel_att
        self.gap         = nn.AdaptiveAvgPool2d(1)
        self.dropout     = nn.Dropout(0.4)
        self.out_dim     = 256

    def forward(self, x):
        x = self.stem(x)
        x = self.block1(x)
        x = self.block2(x)
        x = self.block3(x)
        x = self.spatial_att(x)
        x = self.block4(x)
        x = self.channel_att(x)
        return self.dropout(self.gap(x).flatten(1))

class KeypointEncoder(nn.Module):
    def __init__(self, in_dim=99, out_dim=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, 256), nn.BatchNorm1d(256), nn.GELU(), nn.Dropout(0.3),
            nn.Linear(256, 128),    nn.BatchNorm1d(128), nn.GELU(),
        )
    def forward(self, x): return self.net(x)

class YogaModel(nn.Module):
    def __init__(self, num_classes=8, kp_dim=99):
        super().__init__()
        self.cnn  = CustomCNN()
        self.kp   = KeypointEncoder(in_dim=kp_dim)
        self.head = nn.Sequential(
            nn.Linear(256 + 128, 256), nn.BatchNorm1d(256), nn.GELU(), nn.Dropout(0.4),
            nn.Linear(256, 128), nn.GELU(), nn.Dropout(0.3),
            nn.Linear(128, num_classes)
        )
    def forward(self, img, kp):
        return self.head(torch.cat([self.cnn(img), self.kp(kp)], dim=1))

# ── Verify names match your saved model ───────────────────────────────────
test = CustomCNN()
names = [n for n, _ in test.named_parameters()]
print('spatial_att found:', any('spatial_att' in n for n in names))
print('channel_att found:', any('channel_att' in n for n in names))
print('Model architecture ready!')

spatial_att found: True
channel_att found: True
Model architecture ready!


## Cell 6 — Load model weights from Google Drive

In [7]:
model = YogaModel(num_classes=len(CLASS_NAMES), kp_dim=KP_DIM).to(DEVICE)
ckpt  = torch.load(MODEL_PATH, map_location=DEVICE)
model.load_state_dict(ckpt['model_state_dict'])   # ← correct key
model.eval()

print('Model loaded!')
print(f"Val accuracy : {ckpt.get('val_accuracy', 0)*100:.2f}%")
print(f"Classes      : {ckpt.get('class_names', CLASS_NAMES)}")

Model loaded!
Val accuracy : 96.00%
Classes      : ['adho mukha svanasana', 'virabhadrasana i', 'virabhadrasana ii', 'vriksasana', 'bhujangasana', 'balasana', 'dandasana', 'chaturanga dandasana']


## Cell 7 — Setup MediaPipe + inference pipeline

In [8]:
MP_MODEL = '/content/pose_landmarker.task'
if not os.path.exists(MP_MODEL):
    print('Downloading MediaPipe model...')
    urllib.request.urlretrieve(
        'https://storage.googleapis.com/mediapipe-models/pose_landmarker/pose_landmarker_lite/float16/latest/pose_landmarker_lite.task',
        MP_MODEL
    )

mp_options = mp.tasks.vision.PoseLandmarkerOptions(
    base_options=mp.tasks.BaseOptions(model_asset_path=MP_MODEL),
    running_mode=mp.tasks.vision.RunningMode.IMAGE
)

preprocess = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]),
    ToTensorV2()
])

def extract_keypoints(img_rgb):
    mp_img = mp.Image(image_format=mp.ImageFormat.SRGB, data=img_rgb)
    with mp.tasks.vision.PoseLandmarker.create_from_options(mp_options) as d:
        result = d.detect(mp_img)
    if result.pose_landmarks and len(result.pose_landmarks) > 0:
        lm = result.pose_landmarks[0]
        kp = np.array([[l.x, l.y, l.visibility] for l in lm], dtype=np.float32)
        hip = (kp[23,:2]+kp[24,:2])/2
        sho = (kp[11,:2]+kp[12,:2])/2
        torso = np.linalg.norm(sho-hip)+1e-6
        kp[:,:2] = (kp[:,:2]-hip)/torso
        return kp.flatten()
    return np.zeros(KP_DIM, dtype=np.float32)

def predict_pose(img_rgb):
    inp = preprocess(image=img_rgb)['image'].unsqueeze(0).to(DEVICE)
    kp  = torch.tensor(extract_keypoints(img_rgb)).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        p1 = F.softmax(model(inp, kp), dim=1)
        p2 = F.softmax(model(torch.flip(inp, dims=[3]), kp), dim=1)
        probs = ((p1+p2)/2).squeeze().cpu().numpy()
    return sorted(
        [{'cls': CLASS_NAMES[i], 'prob': float(probs[i])} for i in range(len(CLASS_NAMES))],
        key=lambda x: x['prob'], reverse=True
    )

print('Inference pipeline ready!')

Inference pipeline ready!


## Cell 8 — Build FastAPI server

In [9]:
from fastapi import FastAPI, UploadFile, File, HTTPException, Request
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import JSONResponse, HTMLResponse
import pathlib

app = FastAPI(title='YogaLens API')
app.add_middleware(CORSMiddleware, allow_origins=['*'], allow_methods=['*'], allow_headers=['*'])

# ── Embed the full yogalens.html as a string ──────────────────────────────
# Read the HTML file from Drive (or paste inline below)
HTML_PATH = Path('/content/drive/MyDrive/Yoga-pose-detection ML project/yogalens.html')

def get_html(public_url: str) -> str:
    """Load HTML and auto-inject the API URL so no manual pasting needed."""
    if HTML_PATH.exists():
        html = HTML_PATH.read_text()
    else:
        return '<h2>yogalens.html not found in Drive. Upload it to: ' + str(HTML_PATH) + '</h2>'

    # Auto-inject the ngrok URL so user never needs to paste it
    inject = f"""
<script>
// Auto-injected by server
window.__API_URL__ = '{public_url}/predict';
</script>
"""
    html = html.replace('</head>', inject + '</head>')

    # Also pre-fill the input field and mark as connected
    autofill = """
<script>
document.addEventListener('DOMContentLoaded', function() {
    if(window.__API_URL__) {
        const inp = document.getElementById('api-input');
        const dot = document.getElementById('api-dot');
        if(inp) inp.value = window.__API_URL__;
        if(dot) { dot.classList.remove('err'); dot.classList.add('on'); }
        // Set API_URL directly in the page
        if(typeof API_URL !== 'undefined') API_URL = window.__API_URL__;
        localStorage.setItem('yogalens_api', window.__API_URL__);
    }
});
</script>
"""
    html = html.replace('</body>', autofill + '</body>')
    return html


@app.get('/', response_class=HTMLResponse)
async def serve_ui(request: Request):
    """Serve the Web UI directly — opening the ngrok URL shows your website."""
    base_url = str(request.base_url).rstrip('/')
    return HTMLResponse(content=get_html(base_url))


@app.get('/status')
def status():
    return {'status':'online','model':'YogaLens CNN','classes':len(CLASS_NAMES)}


@app.post('/predict')
async def predict(image: UploadFile = File(...)):
    if not image.content_type.startswith('image/'):
        raise HTTPException(400, 'File must be an image')
    try:
        img_bytes   = await image.read()
        img_rgb     = np.array(Image.open(io.BytesIO(img_bytes)).convert('RGB'))
        predictions = predict_pose(img_rgb)
        top         = predictions[0]
        return JSONResponse({'success':True,'top_class':top['cls'],
                             'confidence':round(top['prob']*100,2),'predictions':predictions})
    except Exception as e:
        raise HTTPException(500, str(e))


print('FastAPI app ready!')
print('Endpoints:')
print('  GET  /        -> serves yogalens.html web UI')
print('  GET  /status  -> health check')
print('  POST /predict -> image upload + prediction')


FastAPI app ready!
Endpoints:
  GET  /        -> serves yogalens.html web UI
  GET  /status  -> health check
  POST /predict -> image upload + prediction


## Cell 9 — START SERVER (keep this cell running!)
**Copy the URL printed below and paste it into your yogalens.html Connect bar**

In [10]:
import nest_asyncio, uvicorn
from pyngrok import ngrok, conf

nest_asyncio.apply()
conf.get_default().auth_token = NGROK_TOKEN

# ── Kill ALL existing tunnels first ───────────────────────────────────────
ngrok.kill()
import time; time.sleep(2)   # wait 2 sec for tunnel to fully close

# Also disconnect any active tunnels via API
try:
    for tunnel in ngrok.get_tunnels():
        ngrok.disconnect(tunnel.public_url)
        print(f'Disconnected: {tunnel.public_url}')
except:
    pass

time.sleep(2)   # wait again before opening new one

# ── Open fresh tunnel ─────────────────────────────────────────────────────
tunnel     = ngrok.connect(8000, proto='http')
PUBLIC_URL = tunnel.public_url

print('=' * 58)
print('   SERVER IS LIVE!')
print('=' * 58)
print(f'   Predict URL : {PUBLIC_URL}/predict')
print()
print('   COPY THIS URL into yogalens.html Connect bar:')
print(f'   >>> {PUBLIC_URL}/predict <<<')
print('=' * 58)
print('   Keep this cell running!')
print('=' * 58)

config = uvicorn.Config(app, host='0.0.0.0', port=8000, loop='asyncio')
server = uvicorn.Server(config)
await server.serve()

INFO:     Started server process [28083]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


   SERVER IS LIVE!
   Predict URL : https://isochronous-bryson-undiagrammatically.ngrok-free.dev/predict

   COPY THIS URL into yogalens.html Connect bar:
   >>> https://isochronous-bryson-undiagrammatically.ngrok-free.dev/predict <<<
   Keep this cell running!
INFO:     43.228.96.61:0 - "GET /predict HTTP/1.1" 405 Method Not Allowed
INFO:     43.228.96.61:0 - "GET /favicon.ico HTTP/1.1" 404 Not Found
INFO:     43.228.96.61:0 - "GET / HTTP/1.1" 200 OK


INFO:     Shutting down
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.
INFO:     Finished server process [28083]
